# Async & asyncio — A Primer (precursor to Parallelization)

**Week 2 — Agentic AI: Building Autonomous Intelligent Systems**

Before an agent can run several LLM calls at once (the **Parallelization** lab), you need one foundational skill: **`asyncio`**. This is a gentle, hands-on trainer for it — **pure Python, no API key, no model**. We use `asyncio.sleep()` to stand in for slow I/O (a network call, a disk read) so you can watch the mechanics directly and for free.

The single idea to take away: when work is spent **waiting** (on the network, a file, a timer), `asyncio` lets many waits **overlap**, so the whole batch finishes in about the time of the slowest one instead of the sum of all of them.

## Introduction

This lesson answers:

- What is a **coroutine**, and what do `async` / `await` do?
- What is the **event loop**?
- How do you run tasks **in sequence** vs **concurrently** — and how much faster is concurrent?
- Why does concurrency speed up *I/O-bound* work but not *CPU-bound* work?

## Learning Goals

After completing this lesson you will be able to:

- Define **coroutine**, `await`, and the **event loop**.
- Run awaitables **sequentially** and measure the time.
- Run them **concurrently** with `asyncio.gather()` and measure the speedup.
- Explain *why* `gather` overlaps the waiting, and when concurrency does not help.

## What is asyncio?

`asyncio` is Python's library for **concurrency on a single thread**.

- A function defined with `async def` is a **coroutine** — a function you can *pause*. Calling it does nothing on its own; it returns a coroutine object that only runs when **awaited**.
- `await something` runs that something and **suspends** the current coroutine until it finishes — handing control back to the **event loop** so other coroutines can make progress in the meantime.
- The **event loop** is the scheduler that juggles all the paused coroutines, waking each one up when the thing it was waiting for is ready.

> **Concurrency is not parallelism.** `asyncio` does not use multiple CPU cores. It interleaves tasks *while they wait on I/O*. That is exactly right for network-bound work (like LLM calls), and does nothing for CPU-bound work (heavy math) — for that you would use multiprocessing.

```
   one thread, one event loop

   coroutine A  --await sleep-->  (paused, waiting)
                                       |  event loop runs someone else
   coroutine B  --await sleep-->  (paused, waiting)
                                       |
   ... when A's wait is done, the loop resumes A ...
```

## Setup

Nothing to install — `asyncio` and `time` are part of Python's standard library.

> **Top-level `await` in Colab.** Colab and Jupyter already run an event loop, so you can write `await ...` directly in a cell (as we do below). In a plain `.py` script there is no running loop, so you would start one with `asyncio.run(main())`. Calling `asyncio.run(...)` inside Colab raises *"event loop is already running"*.

In [45]:
import asyncio
import time

## A pretend "slow" task

`io_task` simulates a job that spends most of its time **waiting** — like calling an API. The `await asyncio.sleep(delay)` is where it hands control back to the event loop. (Swap `asyncio.sleep` for a real network call and nothing else about the structure changes.)

Notice the **`delay` is a parameter** — the *caller* decides how long each task takes. We pass a different delay to Task A, B, and C so you can watch how their waits overlap.

In [46]:
async def io_task(name, delay):
    """Pretend to do slow I/O: announce, wait `delay` seconds (passed in by the caller), then finish."""
    print(f"  {name}: start")
    await asyncio.sleep(delay)   # stand-in for slow I/O; yields control to the event loop
    print(f"  {name}: done after {delay}s")
    return f"{name} finished"

## Running in sequence

Awaiting one task, then the next, runs them **one at a time** — the waits stack up. Notice in the output that Task A fully finishes before Task B starts. The total is roughly the **sum** of every task's delay.

In [47]:
start = time.perf_counter()

await io_task("Task A", 2.0)
await io_task("Task B", 3.0)
await io_task("Task C", 2.5)

print(f"\nSEQUENTIAL total: {time.perf_counter() - start:.2f}s")

  Task A: start
  Task A: done after 2.0s
  Task B: start
  Task B: done after 3.0s
  Task C: start
  Task C: done after 2.5s

SEQUENTIAL total: 7.51s


## Running concurrently

`asyncio.gather(*coroutines)` launches all the coroutines on the event loop **at once** and waits for them together. Now all three print `start` immediately, and each prints `done` only when its own timer elapses — so the total is roughly the time of the **slowest single task**, not the sum.

`gather` returns the results **in the order you passed them in** (not the order they finished), so `results[0]` is always Task A.

In [48]:
start = time.perf_counter()

results = await asyncio.gather(
    io_task("Task A", 2.0),
    io_task("Task B", 3.0),
    io_task("Task C", 2.5),
    io_task("Task D", 4.0)
)

print(f"\nresults: {results}")
print(f"CONCURRENT total: {time.perf_counter() - start:.2f}s")

  Task A: start
  Task B: start
  Task C: start
  Task D: start
  Task A: done after 2.0s
  Task C: done after 2.5s
  Task B: done after 3.0s
  Task D: done after 4.0s

results: ['Task A finished', 'Task B finished', 'Task C finished', 'Task D finished']
CONCURRENT total: 4.00s


## Why it is faster

The speedup comes entirely from **overlapping the waiting** — no code ran faster and no extra CPU core was used. The single thread simply stopped sitting idle.

```
SEQUENTIAL  (await one task, then the next -- the waits stack up)
  A  |==== 2.0s ====|
  B                  |======= 3.0s =======|
  C                                        |===== 2.5s =====|
     +-------------------------------------------------------+  total ~ 7.5s  (the sum)

CONCURRENT  (asyncio.gather -- all launched together, the waits overlap)
  A  |==== 2.0s ====|
  B  |======= 3.0s =======|
  C  |===== 2.5s =====|
     +--------------------+  total ~ 3.0s  (the slowest task alone)
```

## Considerations

- **I/O-bound vs CPU-bound.** `asyncio` only helps when tasks spend time *waiting*. If `io_task` did heavy math instead of `await asyncio.sleep(...)`, concurrency would not help at all — the single thread can only compute one thing at a time. (Try the last exercise to see this.)
- **Order vs timing.** `gather` preserves *input* order in its results list, but tasks finish at unpredictable times — never assume which one finishes first.
- **One failure can stop `gather`.** By default, if any coroutine raises, `gather` raises. Pass `return_exceptions=True` to collect errors as values instead and keep the good results.
- **Top-level `await` is a notebook convenience.** In real scripts, wrap your entry point in `asyncio.run(main())`.

####**3.Sequence a lis**t- Given jobs = [("A", 2.0), ("B", 3.0), ("C", 2.5)], run them sequentially with a for loop, then concurrently with a list comprehension inside gather. Compare the times.

In [49]:
async def io_seq_con_task(name, delay):
    """Pretend to do slow I/O: announce, wait `delay` seconds (passed in by the caller), then finish."""
    print(f"  {name}: start")
    #await asyncio.sleep(delay)   # stand-in for slow I/O; yields control to the event loop
    #Replace await asyncio.sleep(delay) with a busy loop like sum(i * i for i in range(2_000_000))
    sum(i * i for i in range(2_000_000))
    print(f"  {name}: done after {delay}s")
    return f"{name} finished"

#Using For Loop
start = time.perf_counter()
jobs=[("A", 2.0), ("B", 3.0), ("C", 2.5)]
for job,delay in jobs:
  await io_seq_con_task(job,delay)
print(f"\nSEQUENTIAL total: {time.perf_counter() - start:.2f}s")

#Using List inside Gather
start=time.perf_counter()
all_jobs=[("A", 2.0), ("B", 3.0), ("C", 2.5)]
jb=[io_seq_con_task(job, delay) for job, delay in all_jobs]
lst = await asyncio.gather(*jb)
print(f"\nCONCURRENT total: {time.perf_counter() - start:.2f}s")

result=['A finished', 'B finished', 'C finished']

if lst == result:
  print("All input jobs: ",lst,"\n Gathered jobs: ",result)
  print("Order is matched")
else:
  print("All input jobs: ",lst,"\n Gathered jobs: ",result)
  print("Order not matched")


  A: start
  A: done after 2.0s
  B: start
  B: done after 3.0s
  C: start
  C: done after 2.5s

SEQUENTIAL total: 0.50s
  A: start
  A: done after 2.0s
  B: start
  B: done after 3.0s
  C: start
  C: done after 2.5s

CONCURRENT total: 0.52s
All input jobs:  ['A finished', 'B finished', 'C finished'] 
 Gathered jobs:  ['A finished', 'B finished', 'C finished']
Order is matched


### **Questions**
####1. Predict, then run. Before running the cells, compute the sequential and concurrent totals by hand for delays 2.0, 3.0, 2.5. Did the measured times match?-
==> Sequential Time: 2.0 + 3.0 + 2.5 =7.50 seconds (each task waits out its full delay duration one after the other).
Concurrent Time: 2.0, 3.0, 2.5, 4.0 =4.00 seconds (all tasks wait at the same time; 4.00 seconds takes time).

####2. Add a task. Add io_task("Task D", 4.0) to the gather call. Does the concurrent total change to track the new slowest task?-
==> Yes the concurrent total changes from 3.0 secons to 4.0 seconds.

####5. Break it on purpose (CPU-bound). Replace await asyncio.sleep(delay) with a busy loop like sum(i * i for i in range(2_000_000)) and re-run sequential vs concurrent. Why is there no speedup this time?-
==> Since it has no await, Job A will never pause until its loop completes. Then the later jobs will start resulting to consuming more time.

## Where this goes next

In the **Parallelization** lab (`labs/lab02_B_parallelization.ipynb`), `io_task` becomes a real **LLM expert call**, and `asyncio.gather` **fans out** three experts at once instead of waiting for each in turn — the exact pattern you just learned, applied to agents:

```
   this lab                          parallelization lab
   io_task("Task A", ...)     -->    call_expert(technical_expert, topic)
   io_task("Task B", ...)     -->    call_expert(business_strategist, topic)
   asyncio.gather(...)        -->    asyncio.gather(...)   # fan-out
```

## Your turn (exercises)

1. **Predict, then run.** Before running the cells, compute the sequential and concurrent totals by hand for delays `2.0, 3.0, 2.5`. Did the measured times match?
2. **Add a task.** Add `io_task("Task D", 4.0)` to the `gather` call. Does the concurrent total change to track the new slowest task?
3. **Sequence a list.** Given `jobs = [("A", 2.0), ("B", 3.0), ("C", 2.5)]`, run them sequentially with a `for` loop, then concurrently with a list comprehension inside `gather`. Compare the times.
4. **Use the results.** `gather` returns a list — print each returned string and confirm the order matches the order you passed the tasks in.
5. **Break it on purpose (CPU-bound).** Replace `await asyncio.sleep(delay)` with a busy loop like `sum(i * i for i in range(2_000_000))` and re-run sequential vs concurrent. Why is there no speedup this time?

When you're done, save a copy (**File -> Save a copy in Drive**) and submit your notebook link via Canvas.